# 6. lime: from a circuit to DRAM commands

Every DRAM command sequence so far came from **lime**, a compiler from *logic networks* to
programs for logic-in-memory hardware. cinm calls it inside `cinm-opt`, once per primitive.
Here we use it on its own: write a small circuit with **mockturtle** (the logic-synthesis
library lime builds on), compile it with lime's command-line tool, and read the program.

lime knows several architectures; for SIMDRAM its model of a subarray has four kinds of cell:

| cell | what it is |
|---|---|
| `D[i]` | data rows: the circuit's inputs and outputs, and scratch rows |
| `T[0]`…`T[3]` | the four compute rows |
| `DCC[0]`, `DCC[1]` | dual-contact cells: readable as themselves or, written `!DCC`, as their complement |
| `bool[0]`, `bool[1]` | rows held at 0 and 1 |

and three instructions, which are the commands of the earlier notebooks under other names:

| lime | what it does | in cinm |
|---|---|---|
| `RC(x) -> (y, …)` | copies row *x* into one or more rows at once | `AAP` |
| `RC_INV(DCC[i]) -> (…)` | copies a cell's complement, read through its other contact | `AAP` from `DCC0N`/`DCC1N` |
| `TRA(x, y, z)` | activates three rows together: all three hold their majority | `AP` |
| `TRA(x, y, z) -> (w)` | the same, and copies the majority on | `AAP` from the group |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

LIME = pud.ARTIFACTS / "lime"
LIME.mkdir(parents=True, exist_ok=True)

## 1. A circuit, written with mockturtle

mockturtle is a C++ header library. This program builds a 2-bit adder as a
*majority-inverter graph* — the network type lime compiles for SIMDRAM, whose gates are
three-input majorities with optional inversions — and writes it out as Verilog.

In [ ]:
%%writefile ../artifacts/lime/adder2.cpp
// A 2-bit adder as a majority-inverter graph (MIG), written with mockturtle.
#include <cstdio>
#include <mockturtle/generators/arithmetic.hpp>
#include <mockturtle/io/write_dot.hpp>
#include <mockturtle/io/write_verilog.hpp>
#include <mockturtle/networks/mig.hpp>

int main() {
  mockturtle::mig_network mig;

  // Inputs: a = (a1 a0), b = (b1 b0).
  auto a0 = mig.create_pi(), a1 = mig.create_pi();
  auto b0 = mig.create_pi(), b1 = mig.create_pi();

  // Two full adders, the carry of the first feeding the second.
  auto [s0, c0] = mockturtle::full_adder(mig, a0, b0, mig.get_constant(false));
  auto [s1, c1] = mockturtle::full_adder(mig, a1, b1, c0);

  // Outputs: the 3-bit sum (c1 s1 s0).
  mig.create_po(s0);
  mig.create_po(s1);
  mig.create_po(c1);

  mockturtle::write_verilog(mig, "adder2.verilog");   // what lime reads
  mockturtle::write_dot(mig, "adder2.dot");           // to draw it
  std::printf("%u inputs, %u outputs, %u majority gates\n",
              mig.num_pis(), mig.num_pos(), mig.num_gates());
}

In [ ]:
src = LIME / "adder2.cpp"
pud.compile_mockturtle(src, LIME / "adder2")
print(pud.run([LIME / "adder2"], cwd=LIME).stdout)
pud.show(LIME / "adder2.verilog", language="verilog")

Every `assign` with three terms is a majority gate. Drawn, with the inputs at the bottom
(`x0 x1` are `a0 a1`, `x2 x3` are `b0 b1`) and the three outputs at the top — a dashed
edge is an inversion:

In [ ]:
pud.draw(LIME / "adder2.dot")

## 2. Compiling it with lime

```
lime_program_export <circuit> <arch> <mode> <candidates> <rewriting> <size factor> [convention]
```

| argument | values | meaning |
|---|---|---|
| circuit | a `.verilog`, `.aig` or `.pla` file, or a built-in name (`fa`, `mux`, `gt`, …) | what to compile |
| arch | `simdram`, `ambit`, `plim`, `imply`, `felix` | the hardware model |
| mode | `exhaustive`, `greedy` | search for the shortest program, or take the first good one |
| candidates | `all`, `plim_compiler` | which gates may be computed next at each step |
| rewriting | `none`, `compiling`, `greedy`, `lp`, `kbest<n>`, … | whether lime restructures the network first |
| size factor | a number | how far rewriting may grow the network |
| convention | `none`, `carry-in`, `loop-carry` | for the full adder: keep the carry in a cell (section 4) |

lime also *validates* its result: it simulates the program and checks it computes the
network. `validated=1` means it does.

In [ ]:
p = pud.lime(LIME / "adder2.verilog", "simdram", "exhaustive", "all", "none", 0)
pud.show_code(p.program, "text")
print(f"{p.instructions} instructions, {p.cells} cells, validated={p.validated}, {p.seconds:.1f} s")

Read it top to bottom: the inputs are in `D[0]`…`D[3]` (`input_cells`), each instruction
copies rows or activates three, and the outputs end in the rows `output_cells` names.
Rows are reused as soon as their value is dead, which is why a 3-output circuit fits in
14 cells.

## 3. What the arguments change

In [ ]:
circuit = LIME / "adder2.verilog"
variants = [
    ("simdram", "exhaustive", "all",           "none",      0),
    ("simdram", "greedy",     "all",           "none",      0),
    ("simdram", "exhaustive", "plim_compiler", "none",      0),
    ("simdram", "exhaustive", "all",           "compiling", 1),
    ("ambit",   "exhaustive", "all",           "none",      0),
    ("plim",    "exhaustive", "all",           "none",      0),
]
print(f"{'arch':8} {'mode':10} {'candidates':13} {'rewriting':10} {'instr':>5} {'cells':>5} {'cost':>6} {'time':>7}")
for v in variants:
    r = pud.lime(circuit, *v)
    print(f"{v[0]:8} {v[1]:10} {v[2]:13} {v[3]:10} {r.instructions:5} {r.cells:5} {r.cost:6.2f} {r.seconds:6.2f}s")

- **exhaustive vs greedy** is the classic trade-off: the search finds a shorter program
  (fewer commands, so less DRAM time for every bit of every operand) and takes far longer
  to find it. cinm uses the exhaustive search, once per primitive, when it compiles.
- **candidate selection** limits which gates the search considers next; restricting it is
  faster and can cost instructions.
- **rewriting** lets lime restructure the network before compiling it. On a circuit this
  small, mockturtle's own pre-optimisation has already done the work.
- **other architectures** have other instruction sets and cost models, so their counts are
  not comparable one to one: PLiM computes one majority per instruction on a resistive
  memory, and lime's cost for Ambit is not a plain instruction count.

## 4. The full adder, and a carry that stays put

cinm's full adder is lime's built-in `fa`. Adding numbers bit by bit, the carry-out of one
bit is the carry-in of the next, so where the carry lives matters: the **convention**
argument tells lime to take the carry-in from, and leave the carry-out in, the
dual-contact cell `DCC[1]`.

In [ ]:
for convention in ("none", "carry-in", "loop-carry"):
    r = pud.lime("fa", convention=convention)
    print(f"{convention:10} {r.instructions} instructions, {r.cells} cells")
fa = pud.lime("fa", convention="loop-carry")
pud.show_code(fa.program, "text")

Keeping the carry in a cell saves an instruction per bit, 8 instead of 9. That is the
sequence cinm uses for every bit of every addition — compare it with its own listing:

In [ ]:
print(pud.primitive_table(names=("row_add",)))

Same eight steps: `RC(D[0]) -> (T[0], T[3])` is `AAP(T0_T3 <- in0)`, `TRA(DCC[1], T[0], T[2])`
is `AP(DCC1_T0_T2)`, and so on. The convention is what cinm's `pud.resident = "DCC1"`
attribute sets in the next notebook, where the networks cinm hands to lime are
written as MLIR.

> **Try it:** change `adder2.cpp` — a 3-bit adder, a subtractor
> (`mockturtle::carry_ripple_subtractor_inplace` is in the same header), or a single
> `mig.create_maj(a, b, c)` — and compile it again. Or run a built-in circuit:
> `pud.lime("mux")`, `pud.lime("gt")`, `pud.lime("relu")`.